In [ ]:
from google.colab import drive
drive.mount('/content/drive')

1. Import & Seed

In [ ]:
import os
import math
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    recall_score,
    confusion_matrix
)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

2. Load Data

In [ ]:
DATA_DIR = " "  # 본인 경로에 맞게 수정

C_age = np.load(os.path.join(DATA_DIR, "C_age.npy"))
C_sex = np.load(os.path.join(DATA_DIR, "C_sex.npy"))
C_edu = np.load(os.path.join(DATA_DIR, "C_edu.npy"))
S_MMSE = np.load(os.path.join(DATA_DIR, "S_MMSE.npy"))

X_SNP = np.load(os.path.join(DATA_DIR, "X_SNP 1.npy"))
X_GM = np.load(os.path.join(DATA_DIR, "X_GM.npy"))
Y_dis = np.load(os.path.join(DATA_DIR, "Y_dis.npy"))

y_all = np.argmax(Y_dis, axis=1)

print("C_age:", C_age.shape)
print("C_sex:", C_sex.shape)
print("C_edu:", C_edu.shape)
print("S_MMSE:", S_MMSE.shape)
print("X_SNP:", X_SNP.shape)
print("X_GM:", X_GM.shape)
print("Y_dis:", Y_dis.shape)
print("Class counts:", np.bincount(y_all))

3. Clinical Feature 구성

In [ ]:
sex_encoder = LabelEncoder()
C_sex_encoded = sex_encoder.fit_transform(C_sex)

X_clinical = np.stack(
    [C_age, C_sex_encoded, C_edu, S_MMSE],
    axis=1
).astype(np.float32)

X_SNP = X_SNP.astype(np.float32)
X_GM = X_GM.astype(np.float32)

print("Clinical:", X_clinical.shape)
print("SNP:", X_SNP.shape)
print("GM:", X_GM.shape)

4. Binary Task 정의

In [ ]:
TASKS = {
    "CN_vs_EMCI": {
        "positive": [1],
        "negative": [0]
    },
    "CN_vs_MCI": {
        "positive": [1, 2],
        "negative": [0]
    },
    "MCI_vs_AD": {
        "positive": [3],
        "negative": [1, 2]
    },
    "LMCI_vs_AD": {
        "positive": [3],
        "negative": [2]
    },
    "CN_vs_AD": {
        "positive": [3],
        "negative": [0]
    }
}

CLASS_LABEL_MAP = {
    0: "CN",
    1: "EMCI",
    2: "LMCI",
    3: "AD"
}


def make_binary_task(task_name, X_clinical, X_snp, X_gm, y_all):
    task = TASKS[task_name]

    pos_classes = task["positive"]
    neg_classes = task["negative"]

    selected_idx = np.isin(y_all, pos_classes + neg_classes)

    Xc = X_clinical[selected_idx]
    Xs = X_snp[selected_idx]
    Xg = X_gm[selected_idx]

    # 원래 진단 class 유지
    y_orig_sub = y_all[selected_idx]

    # binary label 생성
    y_bin = np.zeros_like(y_orig_sub, dtype=np.int64)
    y_bin[np.isin(y_orig_sub, pos_classes)] = 1
    y_bin[np.isin(y_orig_sub, neg_classes)] = 0

    return Xc, Xs, Xg, y_bin, y_orig_sub


def build_balanced_folds_from_original_classes(y_orig_sub, n_splits=5, random_state=42):
    unique_classes = np.unique(y_orig_sub)
    per_class_fold_indices = {}

    for cls in unique_classes:
        cls_idx = np.where(y_orig_sub == cls)[0]

        rng = np.random.RandomState(random_state)
        cls_idx = cls_idx.copy()
        rng.shuffle(cls_idx)

        split_idx = np.array_split(cls_idx, n_splits)
        per_class_fold_indices[cls] = split_idx

    fold_splits = []
    all_indices = np.arange(len(y_orig_sub))

    for fold_id in range(n_splits):
        val_parts = []

        for cls in unique_classes:
            val_parts.append(per_class_fold_indices[cls][fold_id])

        val_idx = np.sort(np.concatenate(val_parts))

        train_mask = np.ones(len(y_orig_sub), dtype=bool)
        train_mask[val_idx] = False
        train_idx = all_indices[train_mask]

        fold_splits.append((train_idx, val_idx))

    return fold_splits


def print_fold_distribution(task_name, y_orig_sub, y_bin, fold_splits):
    print("\n" + "=" * 80)
    print(f"Fold distribution: {task_name}")
    print("=" * 80)

    unique_classes = np.unique(y_orig_sub)

    for fold_id, (train_idx, val_idx) in enumerate(fold_splits, start=1):
        y_orig_train = y_orig_sub[train_idx]
        y_orig_val = y_orig_sub[val_idx]

        y_bin_train = y_bin[train_idx]
        y_bin_val = y_bin[val_idx]

        train_orig_str = ", ".join([
            f"{CLASS_LABEL_MAP[c]}={np.sum(y_orig_train == c)}"
            for c in unique_classes
        ])

        val_orig_str = ", ".join([
            f"{CLASS_LABEL_MAP[c]}={np.sum(y_orig_val == c)}"
            for c in unique_classes
        ])

        print(f"\nFold {fold_id}")
        print(f"Train original: {train_orig_str}")
        print(f"Val original  : {val_orig_str}")
        print(f"Train binary  : 0={np.sum(y_bin_train == 0)}, 1={np.sum(y_bin_train == 1)}")
        print(f"Val binary    : 0={np.sum(y_bin_val == 0)}, 1={np.sum(y_bin_val == 1)}")


for task_name in TASKS.keys():
    Xc, Xs, Xg, y, y_orig_sub = make_binary_task(
        task_name,
        X_clinical,
        X_SNP,
        X_GM,
        y_all
    )

    print(
        task_name,
        Xc.shape,
        Xs.shape,
        Xg.shape,
        "binary:",
        np.bincount(y),
        "original:",
        {
            CLASS_LABEL_MAP[c]: int(np.sum(y_orig_sub == c))
            for c in np.unique(y_orig_sub)
        }
    )

5. Dataset

In [ ]:
class ADNIDataset(Dataset):
    def __init__(self, X_clinical, X_snp, X_gm, y):
        self.X_clinical = torch.tensor(X_clinical, dtype=torch.float32)
        self.X_snp = torch.tensor(X_snp, dtype=torch.float32)
        self.X_gm = torch.tensor(X_gm, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return {
            "clinical": self.X_clinical[idx],
            "snp": self.X_snp[idx],
            "gm": self.X_gm[idx],
            "label": self.y[idx]
        }

6. Fold별 Scaling

In [ ]:
def scale_fold_data(Xc_train, Xc_val, Xs_train, Xs_val, Xg_train, Xg_val):
    scaler_c = StandardScaler()
    scaler_s = StandardScaler()
    scaler_g = StandardScaler()

    Xc_train = scaler_c.fit_transform(Xc_train)
    Xc_val = scaler_c.transform(Xc_val)

    Xs_train = scaler_s.fit_transform(Xs_train)
    Xs_val = scaler_s.transform(Xs_val)

    Xg_train = scaler_g.fit_transform(Xg_train)
    Xg_val = scaler_g.transform(Xg_val)

    return Xc_train, Xc_val, Xs_train, Xs_val, Xg_train, Xg_val

7. Concat Baseline Model

In [ ]:
class ConcatMLP(nn.Module):
    def __init__(
        self,
        clinical_dim=4,
        snp_dim=2098,
        gm_dim=93,
        hidden_dim=128,
        num_classes=2
    ):
        super().__init__()

        input_dim = clinical_dim + snp_dim + gm_dim

        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, clinical, snp, gm, return_attn=False):
        x = torch.cat([clinical, snp, gm], dim=1)
        logits = self.classifier(x)

        if return_attn:
            return logits, {}

        return logits

8. Tokenizer

In [ ]:
class SNPTokenizer(nn.Module):
    def __init__(self, snp_dim=2098, num_tokens=64, embed_dim=64):
        super().__init__()

        self.snp_dim = snp_dim
        self.num_tokens = num_tokens
        self.embed_dim = embed_dim

        self.group_size = math.ceil(snp_dim / num_tokens)
        self.padded_dim = self.group_size * num_tokens

        self.token_proj = nn.Linear(self.group_size, embed_dim)

    def forward(self, x):
        # x: [B, 2098]
        B = x.size(0)

        if self.padded_dim > self.snp_dim:
            pad_size = self.padded_dim - self.snp_dim
            pad = torch.zeros(B, pad_size, device=x.device, dtype=x.dtype)
            x = torch.cat([x, pad], dim=1)

        # [B, padded_dim] -> [B, num_tokens, group_size]
        x = x.view(B, self.num_tokens, self.group_size)

        # [B, num_tokens, embed_dim]
        tokens = self.token_proj(x)

        return tokens


class GMTokenizer(nn.Module):
    def __init__(self, gm_dim=93, embed_dim=64):
        super().__init__()

        self.gm_dim = gm_dim
        self.embed_dim = embed_dim

        self.token_proj = nn.Linear(1, embed_dim)

    def forward(self, x):
        # x: [B, 93]
        x = x.unsqueeze(-1)          # [B, 93, 1]
        tokens = self.token_proj(x)  # [B, 93, embed_dim]

        return tokens

9. 1DConv-Pooling

In [ ]:
class Conv1DTokenPooling(nn.Module):
    def __init__(self, embed_dim=64, out_dim=64, kernel_size=3):
        super().__init__()

        padding = kernel_size // 2

        self.pool = nn.Sequential(
            nn.Conv1d(
                in_channels=embed_dim,
                out_channels=out_dim,
                kernel_size=kernel_size,
                padding=padding
            ),
            nn.ReLU(),

            nn.Conv1d(
                in_channels=out_dim,
                out_channels=out_dim,
                kernel_size=kernel_size,
                padding=padding
            ),
            nn.ReLU(),

            nn.AdaptiveMaxPool1d(1)
        )

    def forward(self, x):
        """
        x: [B, tokens, embed_dim]
        return: [B, out_dim]
        """
        x = x.transpose(1, 2)   # [B, embed_dim, tokens]
        x = self.pool(x)        # [B, out_dim, 1]
        x = x.squeeze(-1)       # [B, out_dim]
        return x

9.1 Self-Attention only

In [ ]:
class TokenSelfAttentionFusion(nn.Module):
    def __init__(
        self,
        clinical_dim=4,
        snp_dim=2098,
        gm_dim=93,
        num_snp_tokens=64,
        embed_dim=64,
        num_heads=4,
        num_classes=2
    ):
        super().__init__()

        self.snp_tokenizer = SNPTokenizer(
            snp_dim=snp_dim,
            num_tokens=num_snp_tokens,
            embed_dim=embed_dim
        )

        self.gm_tokenizer = GMTokenizer(
            gm_dim=gm_dim,
            embed_dim=embed_dim
        )

        self.snp_self_attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True
        )

        self.gm_self_attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True
        )

        self.snp_norm = nn.LayerNorm(embed_dim)
        self.gm_norm = nn.LayerNorm(embed_dim)

        self.snp_pool = Conv1DTokenPooling(embed_dim=embed_dim, out_dim=embed_dim)
        self.gm_pool = Conv1DTokenPooling(embed_dim=embed_dim, out_dim=embed_dim)

        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 2 + clinical_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, clinical, snp, gm, return_attn=False):
        snp_tokens = self.snp_tokenizer(snp)  # [B, N_snp, E]
        gm_tokens = self.gm_tokenizer(gm)     # [B, 93, E]

        snp_self, snp_attn = self.snp_self_attn(
            snp_tokens,
            snp_tokens,
            snp_tokens,
            need_weights=True,
            average_attn_weights=False
        )

        gm_self, gm_attn = self.gm_self_attn(
            gm_tokens,
            gm_tokens,
            gm_tokens,
            need_weights=True,
            average_attn_weights=False
        )

        snp_out = self.snp_norm(snp_self + snp_tokens)
        gm_out = self.gm_norm(gm_self + gm_tokens)

        snp_vec = self.snp_pool(snp_out)
        gm_vec = self.gm_pool(gm_out)

        z = torch.cat([snp_vec, gm_vec, clinical], dim=1)
        logits = self.classifier(z)

        if return_attn:
            return logits, {
                "snp_self": snp_attn,
                "gm_self": gm_attn
            }

        return logits

9.2 Cross-Attention only

In [ ]:
class TokenCrossAttentionFusion(nn.Module):
    def __init__(
        self,
        clinical_dim=4,
        snp_dim=2098,
        gm_dim=93,
        num_snp_tokens=64,
        embed_dim=64,
        num_heads=4,
        num_classes=2
    ):
        super().__init__()

        self.snp_tokenizer = SNPTokenizer(
            snp_dim=snp_dim,
            num_tokens=num_snp_tokens,
            embed_dim=embed_dim
        )

        self.gm_tokenizer = GMTokenizer(
            gm_dim=gm_dim,
            embed_dim=embed_dim
        )

        self.snp_to_gm_attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True
        )

        self.gm_to_snp_attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True
        )

        self.snp_norm = nn.LayerNorm(embed_dim)
        self.gm_norm = nn.LayerNorm(embed_dim)

        self.snp_pool = Conv1DTokenPooling(embed_dim=embed_dim, out_dim=embed_dim)
        self.gm_pool = Conv1DTokenPooling(embed_dim=embed_dim, out_dim=embed_dim)

        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 2 + clinical_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, clinical, snp, gm, return_attn=False):
        snp_tokens = self.snp_tokenizer(snp)  # [B, N_snp, E]
        gm_tokens = self.gm_tokenizer(gm)     # [B, 93, E]

        # SNP가 GM을 참조
        # Q = SNP, K = GM, V = GM
        snp_from_gm, snp_to_gm_attn = self.snp_to_gm_attn(
            query=snp_tokens,
            key=gm_tokens,
            value=gm_tokens,
            need_weights=True,
            average_attn_weights=False
        )

        # GM이 SNP를 참조
        # Q = GM, K = SNP, V = SNP
        gm_from_snp, gm_to_snp_attn = self.gm_to_snp_attn(
            query=gm_tokens,
            key=snp_tokens,
            value=snp_tokens,
            need_weights=True,
            average_attn_weights=False
        )

        snp_out = self.snp_norm(snp_from_gm + snp_tokens)
        gm_out = self.gm_norm(gm_from_snp + gm_tokens)

        snp_vec = self.snp_pool(snp_out)
        gm_vec = self.gm_pool(gm_out)

        z = torch.cat([snp_vec, gm_vec, clinical], dim=1)
        logits = self.classifier(z)

        if return_attn:
            return logits, {
                "snp_to_gm": snp_to_gm_attn,
                "gm_to_snp": gm_to_snp_attn
            }

        return logits

9.3 Self + Cross-Attention

In [ ]:
class TokenSelfCrossAttentionFusion(nn.Module):
    def __init__(
        self,
        clinical_dim=4,
        snp_dim=2098,
        gm_dim=93,
        num_snp_tokens=64,
        embed_dim=64,
        num_heads=4,
        num_classes=2
    ):
        super().__init__()

        self.snp_tokenizer = SNPTokenizer(
            snp_dim=snp_dim,
            num_tokens=num_snp_tokens,
            embed_dim=embed_dim
        )

        self.gm_tokenizer = GMTokenizer(
            gm_dim=gm_dim,
            embed_dim=embed_dim
        )

        self.snp_self_attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True
        )

        self.gm_self_attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True
        )

        self.snp_to_gm_attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True
        )

        self.gm_to_snp_attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True
        )

        self.snp_self_norm = nn.LayerNorm(embed_dim)
        self.gm_self_norm = nn.LayerNorm(embed_dim)

        self.snp_cross_norm = nn.LayerNorm(embed_dim)
        self.gm_cross_norm = nn.LayerNorm(embed_dim)

        self.snp_pool = Conv1DTokenPooling(embed_dim=embed_dim, out_dim=embed_dim)
        self.gm_pool = Conv1DTokenPooling(embed_dim=embed_dim, out_dim=embed_dim)

        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 2 + clinical_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, clinical, snp, gm, return_attn=False):
        snp_tokens = self.snp_tokenizer(snp)
        gm_tokens = self.gm_tokenizer(gm)

        # 1) Self-Attention
        snp_self, snp_self_attn = self.snp_self_attn(
            snp_tokens,
            snp_tokens,
            snp_tokens,
            need_weights=True,
            average_attn_weights=False
        )

        gm_self, gm_self_attn = self.gm_self_attn(
            gm_tokens,
            gm_tokens,
            gm_tokens,
            need_weights=True,
            average_attn_weights=False
        )

        snp_self = self.snp_self_norm(snp_self + snp_tokens)
        gm_self = self.gm_self_norm(gm_self + gm_tokens)

        # 2) Cross-Attention
        # SNP가 GM을 참조
        # Q = SNP, K = GM, V = GM
        snp_from_gm, snp_to_gm_attn = self.snp_to_gm_attn(
            query=snp_self,
            key=gm_self,
            value=gm_self,
            need_weights=True,
            average_attn_weights=False
        )

        # GM이 SNP를 참조
        # Q = GM, K = SNP, V = SNP
        gm_from_snp, gm_to_snp_attn = self.gm_to_snp_attn(
            query=gm_self,
            key=snp_self,
            value=snp_self,
            need_weights=True,
            average_attn_weights=False
        )

        snp_out = self.snp_cross_norm(snp_from_gm + snp_self)
        gm_out = self.gm_cross_norm(gm_from_snp + gm_self)

        snp_vec = self.snp_pool(snp_out)
        gm_vec = self.gm_pool(gm_out)

        z = torch.cat([snp_vec, gm_vec, clinical], dim=1)
        logits = self.classifier(z)

        if return_attn:
            return logits, {
                "snp_self": snp_self_attn,
                "gm_self": gm_self_attn,
                "snp_to_gm": snp_to_gm_attn,
                "gm_to_snp": gm_to_snp_attn
            }

        return logits

10. Train / Evaluation Function

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for batch in loader:
        clinical = batch["clinical"].to(device)
        snp = batch["snp"].to(device)
        gm = batch["gm"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model(clinical, snp, gm)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)

    return total_loss / len(loader.dataset)


def evaluate(model, loader, device):
    model.eval()

    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for batch in loader:
            clinical = batch["clinical"].to(device)
            snp = batch["snp"].to(device)
            gm = batch["gm"].to(device)
            labels = batch["label"].to(device)

            logits = model(clinical, snp, gm)
            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = torch.argmax(logits, dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    y_prob = np.array(all_probs)

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")

    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = np.nan

    sensitivity = recall_score(y_true, y_pred, pos_label=1, zero_division=0)

    try:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        specificity = tn / (tn + fp + 1e-8)
    except ValueError:
        specificity = np.nan

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "auc": auc,
        "sensitivity": sensitivity,
        "specificity": specificity
    }

11. Cross Validation Runner

In [ ]:
def build_model(model_name):
    if model_name == "concat":
        return ConcatMLP()

    elif model_name == "self":
        return TokenSelfAttentionFusion(
            num_snp_tokens=64,
            embed_dim=64,
            num_heads=4
        )

    elif model_name == "cross":
        return TokenCrossAttentionFusion(
            num_snp_tokens=64,
            embed_dim=64,
            num_heads=4
        )

    elif model_name == "self_cross":
        return TokenSelfCrossAttentionFusion(
            num_snp_tokens=64,
            embed_dim=64,
            num_heads=4
        )

    else:
        raise ValueError(f"Unknown model_name: {model_name}")


def run_experiment(
    task_name,
    model_name="concat",
    epochs=50,
    batch_size=32,
    lr=1e-3,
    weight_decay=1e-4,
    seed=42,
    save_best_model=False,
    save_dir="./saved_models",
    print_distribution=True
):
    set_seed(seed)

    # y: binary label
    # y_orig_sub: 원래 진단 class
    Xc, Xs, Xg, y, y_orig_sub = make_binary_task(
        task_name,
        X_clinical,
        X_SNP,
        X_GM,
        y_all
    )

    # MLP baseline과 동일한 original class 기준 fold split
    fold_splits = build_balanced_folds_from_original_classes(
        y_orig_sub=y_orig_sub,
        n_splits=5,
        random_state=seed
    )

    if print_distribution:
        print_fold_distribution(
            task_name=task_name,
            y_orig_sub=y_orig_sub,
            y_bin=y,
            fold_splits=fold_splits
        )

    fold_results = []

    if save_best_model:
        os.makedirs(save_dir, exist_ok=True)

    for fold, (train_idx, val_idx) in enumerate(fold_splits, 1):
        Xc_train, Xc_val = Xc[train_idx], Xc[val_idx]
        Xs_train, Xs_val = Xs[train_idx], Xs[val_idx]
        Xg_train, Xg_val = Xg[train_idx], Xg[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        Xc_train, Xc_val, Xs_train, Xs_val, Xg_train, Xg_val = scale_fold_data(
            Xc_train,
            Xc_val,
            Xs_train,
            Xs_val,
            Xg_train,
            Xg_val
        )

        train_dataset = ADNIDataset(
            Xc_train,
            Xs_train,
            Xg_train,
            y_train
        )

        val_dataset = ADNIDataset(
            Xc_val,
            Xs_val,
            Xg_val,
            y_val
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            drop_last=True
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False
        )

        model = build_model(model_name).to(device)

        class_counts = np.bincount(y_train)
        class_weights = len(y_train) / (2.0 * class_counts)
        class_weights = torch.tensor(
            class_weights,
            dtype=torch.float32
        ).to(device)

        criterion = nn.CrossEntropyLoss(weight=class_weights)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay
        )

        best_f1 = -1.0
        best_metrics = None
        best_state_dict = None

        for epoch in range(1, epochs + 1):
            train_loss = train_one_epoch(
                model,
                train_loader,
                optimizer,
                criterion,
                device
            )

            metrics = evaluate(
                model,
                val_loader,
                device
            )

            if metrics["macro_f1"] > best_f1:
                best_f1 = metrics["macro_f1"]
                best_metrics = metrics.copy()
                best_state_dict = {
                    k: v.cpu().clone()
                    for k, v in model.state_dict().items()
                }

        best_metrics["fold"] = fold
        best_metrics["task"] = task_name
        best_metrics["model"] = model_name
        best_metrics["train_n"] = len(train_idx)
        best_metrics["val_n"] = len(val_idx)

        fold_results.append(best_metrics)

        if save_best_model:
            save_path = os.path.join(
                save_dir,
                f"{task_name}_{model_name}_fold{fold}.pt"
            )
            torch.save(best_state_dict, save_path)

        print(
            f"[{task_name} | {model_name}] "
            f"Fold {fold} "
            f"F1={best_metrics['macro_f1']:.4f}, "
            f"AUC={best_metrics['auc']:.4f}, "
            f"ACC={best_metrics['accuracy']:.4f}, "
            f"SEN={best_metrics['sensitivity']:.4f}, "
            f"SPEC={best_metrics['specificity']:.4f}"
        )

    return fold_results

12. Run All Experiments

In [ ]:
all_results = []

models_to_run = ["self", "cross", "self_cross"]

for task_name in TASKS.keys():
    for model_name in models_to_run:
        results = run_experiment(
            task_name=task_name,
            model_name=model_name,
            epochs=50,
            batch_size=32,
            lr=1e-3,
            weight_decay=1e-4,
            seed=42,
            save_best_model=True,
            save_dir="./saved_models",
            print_distribution=True
        )

        all_results.extend(results)

df_results = pd.DataFrame(all_results)

df_results.to_csv(
    "token_attention_fusion_results_original_class_fold.csv",
    index=False
)

summary = df_results.groupby(["task", "model"])[
    ["accuracy", "macro_f1", "auc", "sensitivity", "specificity"]
].agg(["mean", "std"])

summary.to_csv("token_attention_fusion_summary_original_class_fold.csv")

print(summary)

Attention Map

In [ ]:
import os
import random
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

def set_seed_local(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


TARGET_TASK = "CN_vs_AD"
TARGET_MODEL = "cross"
SAVE_DIR = "./saved_models"

N_SPLITS = 5
TOP_K_GM = 20
RANDOM_STATE = 42

FIG_NAME = f"attention_map_{TARGET_TASK}_{TARGET_MODEL}_SNPgroup_to_GM_delta95_top{TOP_K_GM}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


GM_ROI_NAMES = [
    "Lateral ventricle R",
    "Globus pallidus R",
    "Caudate nucleus R",
    "Cuneus L",
    "Nucleus accumbens L",
    "Lateral ventricle L",
    "Caudate nucleus L",
    "Temporal lobe WM R",
    "Occipital lobe WM L",
    "Superior parietal lobule R",
    "Lateral occipitotemporal gyrus R",
    "Entorhinal cortex R",
    "Cuneus R",
    "Insula R",
    "Precentral gyrus R",
    "Medial frontal gyrus L",
    "Globus pallidus L",
    "Putamen R",
    "Subthalamic nucleus R",
    "Occipital lobe WM R",
    "Precuneus L",
    "Superior frontal gyrus L",
    "Postcentral gyrus L",
    "Perirhinal cortex R",
    "Postcentral gyrus R",
    "Lingual gyrus R",
    "Superior temporal gyrus R",
    "Fornix R",
    "Middle frontal gyrus R",
    "Inferior frontal gyrus L",
    "Angular gyrus R",
    "Frontal lobe WM L",
    "Posterior limb internal capsule L",
    "Posterior limb internal capsule R",
    "Superior parietal lobule L",
    "Parietal lobe WM L",
    "Precentral gyrus L",
    "Medial front-orbital gyrus L",
    "Parietal lobe WM R",
    "Parahippocampal gyrus R",
    "Occipital pole R",
    "Inferior temporal gyrus R",
    "Medial front-orbital gyrus R",
    "Superior frontal gyrus R",
    "Putamen L",
    "Parahippocampal gyrus L",
    "Fornix L",
    "Precuneus R",
    "Superior occipital gyrus R",
    "Supramarginal gyrus L",
    "Middle frontal gyrus L",
    "Supramarginal gyrus R",
    "Inferior frontal gyrus R",
    "Temporal lobe WM L",
    "Lateral front-orbital gyrus L",
    "Insula L",
    "Medial frontal gyrus R",
    "Angular gyrus L",
    "Medial occipitotemporal gyrus R",
    "Lateral occipitotemporal gyrus L",
    "Occipital pole L",
    "Lateral front-orbital gyrus R",
    "Cingulate region R",
    "Frontal lobe WM R",
    "Temporal pole R",
    "Nucleus accumbens R",
    "Uncus R",
    "Cingulate region L",
    "Subthalamic nucleus L",
    "Hippocampal formation R",
    "Inferior occipital gyrus L",
    "Anterior limb internal capsule L",
    "Superior temporal gyrus L",
    "Uncus L",
    "Middle occipital gyrus R",
    "Middle temporal gyrus L",
    "Lingual gyrus L",
    "Perirhinal cortex L",
    "Inferior temporal gyrus L",
    "Temporal pole L",
    "Entorhinal cortex L",
    "Inferior occipital gyrus R",
    "Superior occipital gyrus L",
    "Hippocampal formation L",
    "Thalamus L",
    "Amygdala L",
    "Medial occipitotemporal gyrus L",
    "Anterior limb internal capsule R",
    "Middle temporal gyrus R",
    "Corpus callosum",
    "Amygdala R",
    "Middle occipital gyrus L",
    "Thalamus R",
]

assert len(GM_ROI_NAMES) == 93, f"GM ROI name length mismatch: {len(GM_ROI_NAMES)}"

GM_NAMES = GM_ROI_NAMES


def collect_cross_attention_for_task(
    task_name,
    save_dir="./saved_models",
    n_splits=5,
    seed=42
):

    set_seed_local(seed)

    task_output = make_binary_task(
        task_name,
        X_clinical,
        X_SNP,
        X_GM,
        y_all
    )

    if len(task_output) == 5:
        Xc, Xs, Xg, y, y_orig_sub = task_output

        fold_splits = build_balanced_folds_from_original_classes(
            y_orig_sub=y_orig_sub,
            n_splits=n_splits,
            random_state=seed
        )

    else:
        from sklearn.model_selection import StratifiedKFold

        Xc, Xs, Xg, y = task_output
        skf = StratifiedKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=seed
        )
        fold_splits = list(skf.split(Xc, y))

    fold_attn_maps = []

    for fold, (train_idx, val_idx) in enumerate(fold_splits, start=1):
        model_path = os.path.join(
            save_dir,
            f"{task_name}_cross_fold{fold}.pt"
        )

        if not os.path.exists(model_path):
            raise FileNotFoundError(
                f"저장된 모델이 없습니다: {model_path}\n"
                f"12번 실험 셀에서 save_best_model=True로 cross 모델을 먼저 저장해야 합니다."
            )


        Xc_train, Xc_val = Xc[train_idx], Xc[val_idx]
        Xs_train, Xs_val = Xs[train_idx], Xs[val_idx]
        Xg_train, Xg_val = Xg[train_idx], Xg[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]


        Xc_train, Xc_val, Xs_train, Xs_val, Xg_train, Xg_val = scale_fold_data(
            Xc_train,
            Xc_val,
            Xs_train,
            Xs_val,
            Xg_train,
            Xg_val
        )

        val_dataset = ADNIDataset(
            Xc_val,
            Xs_val,
            Xg_val,
            y_val
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=32,
            shuffle=False
        )


        model = build_model("cross").to(device)
        state_dict = torch.load(model_path, map_location=device)
        model.load_state_dict(state_dict)
        model.eval()

        batch_attn_maps = []

        with torch.no_grad():
            for batch in val_loader:
                clinical = batch["clinical"].to(device)
                snp = batch["snp"].to(device)
                gm = batch["gm"].to(device)

                logits, attn_dict = model(
                    clinical,
                    snp,
                    gm,
                    return_attn=True
                )

                # snp_to_gm shape:
                # [B, num_heads, num_snp_group_tokens, num_gm_tokens]
                snp_to_gm = attn_dict["snp_to_gm"]

                # sample 평균 + head 평균
                # 결과: [num_snp_group_tokens, num_gm_tokens]
                attn_map = snp_to_gm.mean(dim=0).mean(dim=0)

                batch_attn_maps.append(attn_map.detach().cpu().numpy())

        # fold 평균
        fold_attn = np.mean(batch_attn_maps, axis=0)
        fold_attn_maps.append(fold_attn)

        print(
            f"[{task_name} | cross] Fold {fold} SNP group-to-GM attention map:",
            fold_attn.shape
        )

    # 전체 fold 평균
    mean_attn = np.mean(fold_attn_maps, axis=0)

    return mean_attn



mean_attn = collect_cross_attention_for_task(
    task_name=TARGET_TASK,
    save_dir=SAVE_DIR,
    n_splits=N_SPLITS,
    seed=RANDOM_STATE
)

print("Final mean attention map shape:", mean_attn.shape)
# expected: [64, 93]


uniform_baseline = 1.0 / mean_attn.shape[1]

delta_attn = mean_attn - uniform_baseline

print("Uniform baseline:", uniform_baseline)
print("Delta attention shape:", delta_attn.shape)
print("Delta min/max:", delta_attn.min(), delta_attn.max())


gm_delta_importance = np.percentile(delta_attn, 95, axis=0)

top_gm_idx = np.argsort(gm_delta_importance)[::-1][:TOP_K_GM]

top_delta_attn_map = delta_attn[:, top_gm_idx]

top_gm_labels = [
    GM_NAMES[i] if i < len(GM_NAMES) else f"GM_{i}"
    for i in top_gm_idx
]

print("\nTop attended GM indices based on 95th percentile above average reference level:")
for rank, idx in enumerate(top_gm_idx, start=1):
    label = GM_NAMES[idx] if idx < len(GM_NAMES) else f"GM_{idx}"
    print(
        f"{rank:02d}. GM index {idx:02d} | {label} | "
        f"importance_95p={gm_delta_importance[idx]:.6f}"
    )



vmax = np.max(np.abs(top_delta_attn_map))
vmin = -vmax

plt.figure(figsize=(11, 6))

im = plt.imshow(
    top_delta_attn_map,
    aspect="auto",
    cmap="RdBu_r",
    vmin=vmin,
    vmax=vmax
)

plt.colorbar(im, label="Relative attention weight")

plt.xlabel("GM ROI")
plt.ylabel("SNP group token")


plt.xticks(
    ticks=np.arange(TOP_K_GM),
    labels=top_gm_labels,
    rotation=60,
    ha="right",
    fontsize=8
)

plt.yticks(
    ticks=np.arange(0, mean_attn.shape[0], 8),
    labels=[f"SNP group {i}" for i in range(0, mean_attn.shape[0], 8)]
)

plt.tight_layout()

plt.savefig(f"{FIG_NAME}.png", dpi=300, bbox_inches="tight")
plt.savefig(f"{FIG_NAME}.pdf", bbox_inches="tight")

plt.show()

print(f"\nSaved:")
print(f"- {FIG_NAME}.png")
print(f"- {FIG_NAME}.pdf")